# 00 - Environment Setup

## CyberForge AI - ML Pipeline Environment Configuration

This notebook sets up the complete environment for the CyberForge AI machine learning pipeline.

### What this notebook does:
1. Validates Python version and system requirements
2. Installs and pins all dependencies
3. Configures GPU/CPU detection
4. Sets up Gemini API connectivity
5. Validates Web Scraper API connection
6. Creates necessary directories

### Prerequisites:
- Python 3.10+ (3.11 recommended)
- Access to Gemini API (API key required)
- Access to WebScrapper.live API

## 1. System Validation

In [ ]:
import sys
import platform
import os
from pathlib import Path

print("=" * 60)
print("CYBERFORGE AI - ENVIRONMENT VALIDATION")
print("=" * 60)

# Python version check
python_version = sys.version_info
print(f"\n✓ Python Version: {python_version.major}.{python_version.minor}.{python_version.micro}")

if python_version.major < 3 or (python_version.major == 3 and python_version.minor < 10):
    raise EnvironmentError("Python 3.10+ is required. Please upgrade your Python installation.")

# System info
print(f"✓ Platform: {platform.system()} {platform.release()}")
print(f"✓ Architecture: {platform.machine()}")
print(f"✓ Processor: {platform.processor() or 'Unknown'}")

# Memory info
try:
    import psutil
    memory = psutil.virtual_memory()
    print(f"✓ Available Memory: {memory.available / (1024**3):.2f} GB / {memory.total / (1024**3):.2f} GB")
except ImportError:
    print("⚠ psutil not installed - memory check skipped")

print("\n" + "=" * 60)

## 2. Install Dependencies

In [ ]:
# Core dependencies with pinned versions for reproducibility
DEPENDENCIES = """
# Core ML/AI
numpy>=1.24.0,<2.0.0
pandas>=2.0.0
scikit-learn>=1.3.0
scipy>=1.11.0

# Deep Learning
torch>=2.0.0
transformers>=4.30.0

# Gemini API
google-generativeai>=0.3.0

# Data Processing
joblib>=1.3.0
tqdm>=4.65.0

# Feature Engineering
tldextract>=5.0.0
validators>=0.22.0
ipaddress>=1.0.23

# Web/API
httpx>=0.25.0
aiohttp>=3.8.0
requests>=2.31.0

# Hugging Face
huggingface_hub>=0.19.0

# Utilities
python-dotenv>=1.0.0
pyyaml>=6.0.0
psutil>=5.9.0
"""

# Write requirements file
requirements_path = Path("../requirements_notebooks.txt")
requirements_path.write_text(DEPENDENCIES.strip())
print(f"✓ Requirements written to: {requirements_path.absolute()}")

In [ ]:
# Install dependencies
import subprocess

print("Installing dependencies... This may take a few minutes.")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("✓ All dependencies installed successfully!")
else:
    print(f"⚠ Installation warnings: {result.stderr[:500] if result.stderr else 'None'}")

## 3. GPU/CPU Detection

In [ ]:
import torch

print("=" * 60)
print("COMPUTE DEVICE DETECTION")
print("=" * 60)

# Check CUDA availability
cuda_available = torch.cuda.is_available()
print(f"\n✓ PyTorch Version: {torch.__version__}")
print(f"✓ CUDA Available: {cuda_available}")

if cuda_available:
    print(f"✓ CUDA Version: {torch.version.cuda}")
    print(f"✓ GPU Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  - GPU {i}: {props.name} ({props.total_memory / (1024**3):.2f} GB)")
    DEVICE = torch.device("cuda")
else:
    print("⚠ No GPU detected - using CPU for training")
    DEVICE = torch.device("cpu")

# Check MPS (Apple Silicon)
if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print("✓ Apple MPS (Metal) available")
    DEVICE = torch.device("mps")

print(f"\n✓ Selected Device: {DEVICE}")
print("=" * 60)

## 4. Environment Variables & API Configuration

In [ ]:
from dotenv import load_dotenv
import os

# Load environment variables from .env file
env_path = Path("../.env")
if env_path.exists():
    load_dotenv(env_path)
    print(f"✓ Loaded environment from: {env_path.absolute()}")
else:
    print(f"⚠ No .env file found at {env_path.absolute()}")

# Configuration class
class Config:
    # API Keys
    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
    HUGGINGFACE_TOKEN = os.getenv("HUGGINGFACE_API_TOKEN", "")
    WEBSCRAPER_API_KEY = "sk-fd14eaa7bceb478db7afc7256e514d2b"  # WebScrapper.live API
    WEBSCRAPER_API_URL = "http://webscrapper.live/api/scrape"
    
    # Paths
    BASE_DIR = Path("..").resolve()
    DATASETS_DIR = BASE_DIR / "datasets"
    MODELS_DIR = BASE_DIR / "models"
    ARTIFACTS_DIR = BASE_DIR / "artifacts"
    
    # ML Settings
    RANDOM_STATE = 42
    TEST_SIZE = 0.2
    CV_FOLDS = 5
    
    # Device
    DEVICE = DEVICE

config = Config()

# Validate required API keys
print("\n" + "=" * 60)
print("API CONFIGURATION STATUS")
print("=" * 60)
print(f"✓ Gemini API Key: {'Configured' if config.GEMINI_API_KEY else '⚠ NOT SET'}")
print(f"✓ HuggingFace Token: {'Configured' if config.HUGGINGFACE_TOKEN else '⚠ NOT SET'}")
print(f"✓ WebScraper API: Configured")

## 5. Gemini API Connectivity Test

In [ ]:
import google.generativeai as genai

def test_gemini_connection():
    """Test Gemini API connectivity"""
    if not config.GEMINI_API_KEY:
        return False, "API key not configured"
    
    try:
        genai.configure(api_key=config.GEMINI_API_KEY)
        model = genai.GenerativeModel('gemini-1.5-flash')
        response = model.generate_content("Respond with only: OK")
        return True, response.text.strip()
    except Exception as e:
        return False, str(e)

print("Testing Gemini API connection...")
success, message = test_gemini_connection()

if success:
    print(f"✓ Gemini API: Connected successfully")
else:
    print(f"⚠ Gemini API: Connection failed - {message}")

## 6. Web Scraper API Connectivity Test

In [ ]:
import httpx

async def test_webscraper_connection():
    """Test WebScrapper.live API connectivity"""
    try:
        async with httpx.AsyncClient(timeout=30.0) as client:
            response = await client.post(
                config.WEBSCRAPER_API_URL,
                json={"url": "https://example.com"},
                headers={
                    "Content-Type": "application/json",
                    "X-API-Key": config.WEBSCRAPER_API_KEY
                }
            )
            if response.status_code == 200:
                return True, "Connected"
            else:
                return False, f"Status {response.status_code}"
    except Exception as e:
        return False, str(e)

print("Testing Web Scraper API connection...")

# Run async test
import asyncio
try:
    loop = asyncio.get_event_loop()
except RuntimeError:
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

success, message = loop.run_until_complete(test_webscraper_connection())

if success:
    print(f"✓ WebScraper API: Connected successfully")
else:
    print(f"⚠ WebScraper API: {message}")

## 7. Create Directory Structure

In [ ]:
# Create necessary directories
directories = [
    config.DATASETS_DIR,
    config.MODELS_DIR,
    config.ARTIFACTS_DIR,
    config.BASE_DIR / "logs",
    config.BASE_DIR / "cache",
]

print("Creating directory structure...")
for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)
    print(f"  ✓ {directory}")

print("\n✓ Directory structure ready!")

## 8. Save Configuration for Other Notebooks

In [ ]:
import json

# Export configuration for other notebooks
notebook_config = {
    "device": str(DEVICE),
    "python_version": f"{python_version.major}.{python_version.minor}.{python_version.micro}",
    "torch_version": torch.__version__,
    "cuda_available": cuda_available,
    "base_dir": str(config.BASE_DIR),
    "datasets_dir": str(config.DATASETS_DIR),
    "models_dir": str(config.MODELS_DIR),
    "artifacts_dir": str(config.ARTIFACTS_DIR),
    "random_state": config.RANDOM_STATE,
    "test_size": config.TEST_SIZE,
    "cv_folds": config.CV_FOLDS,
    "gemini_configured": bool(config.GEMINI_API_KEY),
    "huggingface_configured": bool(config.HUGGINGFACE_TOKEN),
    "created_at": str(pd.Timestamp.now())
}

config_path = config.BASE_DIR / "notebook_config.json"
with open(config_path, "w") as f:
    json.dump(notebook_config, f, indent=2)

print(f"✓ Configuration saved to: {config_path}")
print("\n" + json.dumps(notebook_config, indent=2))

## 9. Environment Summary

In [ ]:
print("\n" + "=" * 60)
print("ENVIRONMENT SETUP COMPLETE")
print("=" * 60)
print(f"""
✅ Python: {python_version.major}.{python_version.minor}.{python_version.micro}
✅ Device: {DEVICE}
✅ PyTorch: {torch.__version__}
✅ Gemini API: {'Ready' if config.GEMINI_API_KEY else 'Not configured'}
✅ HuggingFace: {'Ready' if config.HUGGINGFACE_TOKEN else 'Not configured'}
✅ WebScraper API: Ready
✅ Directories: Created

You can now proceed to the next notebook:
  → 01_data_acquisition.ipynb
""")
print("=" * 60)